# 05. Objects, dunder protocols, dataclasses, slots and design patterns

**Goal.** Model a domain with classes that behave like built-ins, cost little memory and stay easy to change.

**You will be able to:**
1. Implement operators, containers and validation through special methods and descriptors.
2. Choose between plain classes, dataclasses and `__slots__` using measured memory.
3. Apply Strategy, Observer, Registry and State patterns with small, tested code.
4. Read the method resolution order (MRO) and use abstract base classes.

## 1. Dunder protocols

### How it works under the hood
- Syntax maps to special methods on the **type**: `a + b` becomes `type(a).__add__(a, b)`, `len(x)` becomes `type(x).__len__(x)`, `x[i]` becomes `type(x).__getitem__(x, i)`.
- Lookup skips the instance `__dict__` for these methods, so you cannot patch an operator on one object.
- If `__add__` returns `NotImplemented`, Python tries the right operand's `__radd__`. This is how `2 * vector` works.
- Defining `__eq__` sets `__hash__` to `None` unless you define it again. Equal objects must hash equal.
- A class with only `__getitem__` and `__len__` is already iterable, reversible and supports `in`, through the legacy sequence protocol.

### Production use
- Domain types that read naturally: `Money`, `Vector`, `Interval`, `Version`.
- Library objects that plug into `len`, `sorted`, `with`, `in` and `for`: ORM querysets, pandas frames, file-like objects.

### Trade-offs

| Approach | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Dunder methods | Methods live on the class | One Python call per operation | Natural call sites | Works with built-ins and libraries | Easy to break contracts (eq and hash) |
| Named methods (`v.add(w)`) | Same | Same | Explicit but verbose | No surprises | Cannot use operators or built-ins |
| Inherit from `list` or `dict` | Full base storage | C speed for base operations | Familiar | Free behaviour | Leaky: overrides often skipped by C code |

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "understanding_python").is_dir())
sys.path.insert(0, str(ROOT / "src"))
print("python", sys.version.split()[0], "| package root", ROOT.name)

python 3.13.6 | package root understanding-python


In [2]:
from understanding_python.oop import Playlist
from understanding_python.patterns import Vector

v, w = Vector(3, 4), Vector(1, 2)
print(v + w, "| 2 * v =", 2 * v, "| |v| =", abs(v), "| bool(Vector(0, 0)) =", bool(Vector(0, 0)))
print("hash contract:", hash(v) == hash(Vector(3, 4)), "| usable in a set:", len({v, Vector(3, 4), w}))
print("operator lookup is on the type:", Vector.__add__(v, w) == v + w)

try:
    v + 5
except TypeError as exc:
    print("unsupported operand:", exc)


class Meters:
    def __init__(self, n):
        self.n = n

    def __add__(self, other):
        if isinstance(other, Meters):
            return Meters(self.n + other.n)
        return NotImplemented

    def __radd__(self, other):
        return self if other == 0 else NotImplemented

    def __repr__(self):
        return f"Meters({self.n})"


print("sum() starts at 0, so __radd__ runs:", sum([Meters(2), Meters(3.5)]))

songs = Playlist(["Intro", "Halcyon", "Outro", "Encore"])
print("len:", len(songs), "| index:", songs[1], "| slice:", songs[1:3], "| in:", "Outro" in songs)
print("iterates and reverses with only __getitem__ and __len__:", list(songs), list(reversed(songs)))

Vector(4, 6) | 2 * v = Vector(6, 8) | |v| = 5.0 | bool(Vector(0, 0)) = False
hash contract: True | usable in a set: 2
operator lookup is on the type: True
unsupported operand: unsupported operand type(s) for +: 'Vector' and 'int'
sum() starts at 0, so __radd__ runs: Meters(5.5)
len: 4 | index: Halcyon | slice: Playlist(['Halcyon', 'Outro']) | in: True
iterates and reverses with only __getitem__ and __len__: ['Intro', 'Halcyon', 'Outro', 'Encore'] ['Encore', 'Outro', 'Halcyon', 'Intro']


## 2. Dataclasses

### How it works under the hood
- `@dataclass` inspects the class annotations at decoration time and generates source code for `__init__`, `__repr__` and `__eq__`. It runs `exec` once and attaches the functions to the class.
- `frozen=True` generates `__setattr__` and `__delattr__` that raise `FrozenInstanceError`, and a `__hash__`.
- `order=True` adds `__lt__` and friends that compare field tuples in declaration order.
- `slots=True` (3.10+) recreates the class with `__slots__`, so instances carry no `__dict__`.
- `field(default_factory=list)` creates a new list per instance and avoids the shared-default bug.

### Production use
- Immutable records: orders, events, configuration, API payloads.
- Value objects that serve as dictionary keys or set members.
- Typed containers between layers, ahead of heavier validation libraries.

### Trade-offs

| Type | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Plain class | `__dict__` per instance | Slowest attribute access | Most boilerplate | Total control | Hand-written `__init__`, `__eq__`, `__repr__` |
| `@dataclass` | Same as plain | Same | Concise | Generated, consistent methods | No validation by default |
| `@dataclass(frozen=True, slots=True)` | Smallest | Fast reads, slower creation (frozen `__init__` calls `object.__setattr__`) | Concise and safe | Hashable, immutable, compact | Cannot mutate; slots limit inheritance |
| `namedtuple` | Tuple-sized | Fast | Concise | Light, unpackable | Index access allowed, weak typing |

In [3]:
import dataclasses
import timeit
from collections import namedtuple

from understanding_python.oop import Order, OrderLine

order = Order("SO-1042", "ada@example.com", (OrderLine("pen", 3, 1.50), OrderLine("notebook", 2, 4.25)))
print(order)
print("total:", order.total(), "| fields:", [f.name for f in dataclasses.fields(order)])
print("match args:", Order.__match_args__)

try:
    order.customer = "someone@else.com"
except dataclasses.FrozenInstanceError as exc:
    print("frozen:", type(exc).__name__)

bigger = dataclasses.replace(order, lines=order.lines + (OrderLine("ink", 1, 7.0),))
print("replace() builds a new record:", bigger.total(), "| original unchanged:", order.total())
print("hashable:", len({order, bigger, order}), "records in a set")
print("has __dict__:", hasattr(order, "__dict__"))


@dataclasses.dataclass
class LooseLine:
    sku: str
    qty: int
    unit_price: float


Pair = namedtuple("Pair", "sku qty unit_price")
t_slot = timeit.timeit(lambda: OrderLine("pen", 3, 1.5), number=200_000)
t_loose = timeit.timeit(lambda: LooseLine("pen", 3, 1.5), number=200_000)
t_named = timeit.timeit(lambda: Pair("pen", 3, 1.5), number=200_000)
print(f"create 1 record: slots dataclass {t_slot * 5000:.0f} ns | plain dataclass {t_loose * 5000:.0f} ns | namedtuple {t_named * 5000:.0f} ns")

Order(order_id='SO-1042', customer='ada@example.com', lines=(OrderLine(sku='pen', qty=3, unit_price=1.5), OrderLine(sku='notebook', qty=2, unit_price=4.25)))
total: 13.0 | fields: ['order_id', 'customer', 'lines']
match args: ('order_id', 'customer', 'lines')
frozen: FrozenInstanceError
replace() builds a new record: 20.0 | original unchanged: 13.0
hashable: 2 records in a set
has __dict__: False


create 1 record: slots dataclass 736 ns | plain dataclass 208 ns | namedtuple 391 ns


## 3. `__slots__` and per-instance memory

### How it works under the hood
- A normal instance stores attributes in a per-instance `__dict__`. A dictionary costs hundreds of bytes even for two attributes.
- `__slots__ = ("x", "y")` replaces the dict with fixed-offset slots inside the object. Each slot is a `member_descriptor` on the class.
- Assigning an attribute not named in `__slots__` raises `AttributeError`. Instances also lose `__weakref__` unless you list it.
- Since 3.11 CPython shares key tables between instances of one class, which narrows the gap. Slots still win on size and speed.

### Production use
- Millions of small objects: graph nodes, parsed tokens, order lines, market ticks.
- Memory-bound workers where each saved byte multiplies by the object count.

### Trade-offs

| Layout | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| `__dict__` instances | Highest | Hash lookup per access | Default | Dynamic attributes, easy monkey-patching | Memory heavy |
| `__slots__` | Roughly 25 to 30 percent less on 3.13 (more on older versions) | Similar reads, faster creation | One extra line | Compact, typo-safe | No dynamic attributes, careful with multiple inheritance |
| Struct-like arrays (`array`, NumPy) | Smallest | Vectorised | Different style | Bulk numeric data | Not objects, no methods |

In [4]:
import tracemalloc

from understanding_python.oop import PlainPoint, SlotPoint


def measure(cls, n=100_000):
    tracemalloc.start()
    objs = [cls(float(i), float(-i)) for i in range(n)]
    current = tracemalloc.get_traced_memory()[0]
    tracemalloc.stop()
    return current / n, objs


plain_bytes, plain_objs = measure(PlainPoint)
slot_bytes, slot_objs = measure(SlotPoint)
print(f"plain class: {plain_bytes:.0f} bytes per point | slots: {slot_bytes:.0f} bytes per point | saving {100 * (1 - slot_bytes / plain_bytes):.0f}%")
print("has __dict__ :", hasattr(plain_objs[0], "__dict__"), "|", hasattr(slot_objs[0], "__dict__"))
print("slot is a    :", type(SlotPoint.x).__name__)

p = SlotPoint(1.0, 2.0)
try:
    p.z = 3.0
except AttributeError as exc:
    print("typo guard   :", exc)

import timeit

plain_t = timeit.timeit("p.x + p.y", globals={"p": plain_objs[0]}, number=2_000_000)
slot_t = timeit.timeit("p.x + p.y", globals={"p": slot_objs[0]}, number=2_000_000)
print(f"read two attributes: plain {plain_t * 500:.0f} ns | slots {slot_t * 500:.0f} ns")

plain class: 144 bytes per point | slots: 104 bytes per point | saving 28%
has __dict__ : True | False
slot is a    : member_descriptor
typo guard   : 'SlotPoint' object has no attribute 'z' and no __dict__ for setting new attributes


read two attributes: plain 60 ns | slots 59 ns


## 4. Design patterns that earn their place

### How it works under the hood
- **Strategy**: pass behaviour as an object or function. The caller depends on the call shape, not on a class. Python functions are first-class, so a function often replaces a class hierarchy.
- **Observer**: a publisher keeps a list of callables per topic and calls each on an event. It decouples producers from consumers.
- **Registry**: `__init_subclass__` runs when a subclass is defined, so subclasses register themselves with no import-time wiring.
- **State**: a transition table maps each state to its legal next states. Invalid moves raise instead of corrupting data.

### Production use
- Strategy: discount rules, pricing, retry policies, serialisers.
- Observer: domain events, webhooks, UI updates, metrics hooks.
- Registry: payment providers, file format readers, plugin systems.
- State: order lifecycle, subscription status, CI job status.

### Trade-offs

| Pattern | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Strategy | One small object per strategy | One indirect call | Clear extension point | Open for extension, easy to test | More types when a lambda would do |
| Observer | List of callbacks per topic | One call per subscriber | Hidden control flow | Loose coupling | Hard to trace, leaks if never unsubscribed |
| Registry via `__init_subclass__` | One dict | One lookup | Declarative | No central if/else | Import order matters, global state |
| State table | One dict | One lookup | Rules visible in one place | Impossible moves fail fast | Table grows with states |

In [5]:
from understanding_python.oop import OrderStateMachine, PaymentMethod
from understanding_python.patterns import Checkout, EventBus, NoDiscount, Percent

basket = [49.99, 12.50, 7.25]
print("strategy: none", Checkout(NoDiscount()).total(basket), "| 10% off", Checkout(Percent(10)).total(basket))

bus, audit_log, sent = EventBus(), [], []
bus.subscribe("order.paid", audit_log.append)
bus.subscribe("order.paid", lambda e: sent.append(f"receipt for {e['id']}"))
delivered = bus.publish("order.paid", {"id": "SO-1042", "total": 69.74})
print("observer: delivered to", delivered, "subscribers |", sent, "| audit entries:", len(audit_log))
print("no subscribers:", bus.publish("order.refunded", {}))

print("registry:", sorted(PaymentMethod.registry))
for key, amount in (("card", 100.0), ("bank", 100.0)):
    print(f"  {key:5} fee on {amount:.2f} = {PaymentMethod.create(key).fee(amount):.2f}")
try:
    PaymentMethod.create("cash")
except ValueError as exc:
    print("unknown key:", exc)

sm = OrderStateMachine()
for step in ("paid", "shipped", "delivered"):
    sm.move(step)
print("state path:", " -> ".join(sm.history))
try:
    sm.move("cancelled")
except ValueError as exc:
    print("illegal move:", exc)

strategy: none 69.74 | 10% off 62.77
observer: delivered to 2 subscribers | ['receipt for SO-1042'] | audit entries: 1
no subscribers: 0
registry: ['bank', 'card']
  card  fee on 100.00 = 3.20
  bank  fee on 100.00 = 0.80
unknown key: unknown payment method 'cash'; known: ['bank', 'card']
state path: placed -> paid -> shipped -> delivered
illegal move: cannot move from delivered to cancelled


## 5. Descriptors, the MRO and abstract base classes

### How it works under the hood
- A **descriptor** is a class attribute with `__get__` and optionally `__set__`. Attribute access on an instance calls it. Properties, methods, `classmethod` and slots all use this mechanism.
- A data descriptor (defines `__set__`) wins over the instance `__dict__`. A non-data descriptor loses to it.
- The **MRO** is the C3 linearisation of a class and its bases. `super()` follows the MRO, not the parent class alone. `Class.__mro__` shows it.
- An **ABC** records abstract method names. Instantiating a subclass that does not override all of them raises `TypeError`.

### Production use
- Validation attributes in ORMs and form libraries are descriptors.
- Cooperative multiple inheritance (mixins) in Django and logging frameworks relies on the MRO.
- ABCs define plugin contracts that fail at instantiation, not on first call.

### Trade-offs

| Tool | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Descriptor | One object per attribute on the class | Python call per access | Reusable validation | One rule, many classes | Magic for newcomers |
| `@property` | One object per property | Python call per access | Simple for one class | Inline logic | Repeats across classes |
| Mixins and MRO | Class objects only | Method lookup walks the MRO (cached) | Compact reuse | Share behaviour | Diamond bugs if `super()` is skipped |
| ABC | Registry data | Cheap `isinstance` | Explicit contract | Early failure | Forces inheritance |

In [6]:
from understanding_python.oop import LineItem, Positive
from understanding_python.patterns import Circle, Rect, Shape, total_area

item = LineItem("SKU-77", qty=3, unit_price=9.99)
print("subtotal:", item.subtotal, "| descriptor lives on the class:", type(LineItem.__dict__["qty"]).__name__)
for field, bad in (("qty", 0), ("unit_price", -1.0)):
    try:
        setattr(item, field, bad)
    except ValueError as exc:
        print("rejected:", exc)
print("value stored on the instance:", {k: v for k, v in vars(item).items()})


class Auditable:
    def save(self):
        return ["Auditable.save"] + super().save()


class Cached:
    def save(self):
        return ["Cached.save"] + super().save()


class Base:
    def save(self):
        return ["Base.save"]


class OrderService(Auditable, Cached, Base):
    pass


print("MRO:", [c.__name__ for c in OrderService.__mro__])
print("cooperative super():", OrderService().save())

try:
    Shape()
except TypeError as exc:
    print("abstract class:", exc)
print("total area:", round(total_area([Rect(2, 3), Circle(1.5)]), 3))

subtotal: 29.97 | descriptor lives on the class: Positive
rejected: qty must be positive, got 0
rejected: unit_price must be positive, got -1.0
value stored on the instance: {'sku': 'SKU-77', '_qty': 3, '_unit_price': 9.99}
MRO: ['OrderService', 'Auditable', 'Cached', 'Base', 'object']
cooperative super(): ['Auditable.save', 'Cached.save', 'Base.save']
abstract class: Can't instantiate abstract class Shape without an implementation for abstract method 'area'
total area: 13.069
